# Notebook 01 — Tracking Pipeline

This notebook walks through every stage of the tracking pipeline interactively.
It is equivalent to running `scripts/run_tracking.py` but lets you inspect
intermediate outputs at each step.

## Stages
1. Setup & configuration
2. (Optional) Background subtraction
3. Draw vial ROIs
4. RF-DETR + OC-SORT tracking → wide CSV
5. Hungarian stitching → stitched long CSV
6. Vial assignment + compact IDs → compact_tracks.csv
7. Overlay video rendering

**Replace all `PLACEHOLDER` paths with your actual file paths.**

In [ ]:
import sys
sys.path.insert(0, '..')   

import json
import os
import re
import cv2
import yaml
import pandas as pd
from pathlib import Path
from IPython.display import Video

from src.preprocessing import preprocess_bgsub_gui
from src.metrics import run_diagnostics
from src.tracking import export_tracks_xy_tuple_csv_one_config
from src.stitching import wide_to_long, build_tracklets, stitch
from src.roi import draw_and_save_vial_rois, assign_vials_and_compact_ids
from src.visualization import render_vial_overlay_video, render_raw_overlay_video
from utils import save_run_params

## 1 — Configuration

Set your paths and Roboflow credentials here.

In [ ]:
# ---- EDIT THESE ----
RAW_VIDEO = r"C:\Users\emmav\Downloads\superfly\2024-02-05_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m\31 DPE\001\2024-03-01_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_31d_001-converted.mp4"
MODEL_ID  = "flies-123/1"   # e.g. "flies-123/1"

# Load API key from creds_config.yaml (not committed to git)
with open("../creds_config.yaml", "r") as f:
    creds_config = yaml.safe_load(f)
API_KEY = creds_config["API_KEY"]

# Load defaults from config.yaml (override below if needed)
with open("../config.yaml") as _f:
    _cfg = yaml.safe_load(_f)
_t = _cfg.get("tracker", {})
_s = _cfg.get("stitching", {})
_p = _cfg.get("preprocessing", {})

confidence              = _t.get("confidence", 0.1)
lost_track_buffer       = _t.get("lost_track_buffer", 90)
min_matching_threshold  = _t.get("minimum_matching_threshold", 0.2)
min_consecutive_frames  = _t.get("minimum_consecutive_frames", 3)
asso_func               = _t.get("asso_func", "diou")
vial_count_cap          = _s.get("vial_count_cap", 7)
bg_gain                 = _p.get("bg_gain", 1.2)
bg_white_level          = _p.get("bg_white_level", 245)
bg_percentile           = _p.get("bg_percentile", 85.0)
bg_sample_stride        = _p.get("bg_sample_stride", 1)
default_end             = _p.get("default_end", 700)

# Auto-increment output directory: run_1, run_2, run_3, ...
_outputs_root = Path("../outputs")
_outputs_root.mkdir(parents=True, exist_ok=True)
_existing = [d for d in _outputs_root.iterdir() if d.is_dir() and d.name.startswith("run_")]
_next_n = max((int(d.name.split("_")[1]) for d in _existing if d.name.split("_")[1].isdigit()), default=0) + 1
OUTPUT_PATH = str(_outputs_root / f"run_{_next_n}")

os.makedirs(OUTPUT_PATH, exist_ok=True)
PATH_TO_VID = RAW_VIDEO

# Extract a short label from the filename: e.g. "2024-03-01_..._31d_001-converted" → "31d_n001"
# Pattern: _<age>d_<3-digit video number> anywhere in the stem.
_raw_stem = Path(RAW_VIDEO).stem
_m = re.search(r'_(\d+d)_(\d{3})', _raw_stem)
short_name = f"{_m.group(1)}_n{_m.group(2)}" if _m else _raw_stem

print("Output dir:", OUTPUT_PATH)
print("Short name:", short_name)
print(f"asso_func={asso_func}, vial_count_cap={vial_count_cap}")
print(f"bg_gain={bg_gain}, bg_white_level={bg_white_level}, bg_percentile={bg_percentile}, bg_sample_stride={bg_sample_stride}, default_end={default_end}")
_cap = cv2.VideoCapture(RAW_VIDEO)
save_run_params(OUTPUT_PATH, "config", {
    "video": RAW_VIDEO, "output_dir": OUTPUT_PATH, "short_name": short_name,
    "video_fps": _cap.get(cv2.CAP_PROP_FPS),
    "video_width": int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "video_height": int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    "video_frames": int(_cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    "tracker": {"confidence": confidence, "lost_track_buffer": lost_track_buffer,
                 "min_matching_threshold": min_matching_threshold,
                 "min_consecutive_frames": min_consecutive_frames, "asso_func": asso_func},
    "preprocessing": {"bg_gain": bg_gain, "bg_white_level": bg_white_level,
                       "bg_percentile": bg_percentile, "bg_sample_stride": bg_sample_stride},
})
_cap.release()

## 2 — (Optional) Background subtraction

Opens a GUI: draw a crop ROI and choose a frame range.
The output is a `_pp.mp4` file with the **85th-percentile** background subtracted.
Skip this cell if your video already has good contrast.

In [4]:
preprocess = True  # set to True to run the GUI
RAW_VIDEO = r"../2024-02-05_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m/13 DPE/001/2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_001-converted.mp4"

if preprocess:
    pp_out = os.path.join(OUTPUT_PATH, Path(RAW_VIDEO).stem + "_pp.mp4")
    PATH_TO_VID = Path(
        preprocess_bgsub_gui(
            video_path=RAW_VIDEO,
            out_mp4=pp_out,
            default_end=default_end,
            gain=bg_gain,
            white_level=bg_white_level,
            bg_sample_stride=bg_sample_stride,
            bg_percentile=bg_percentile,
        )
    )
    print("Preprocessed video:", PATH_TO_VID)
save_run_params(OUTPUT_PATH, "preprocessing", {"video_pp": str(PATH_TO_VID)})

Saved bgsub video: ..\outputs\run_44\2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_001-converted_pp.mp4
Background (85.0th percentile) from 312 frames (stride=1).
Preprocessed video: ..\outputs\run_44\2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_001-converted_pp.mp4


## 3 — Draw vial ROIs

Opens an OpenCV GUI on frame 0: drag rectangles around each vial.
Press **q** when all 6 ROIs are drawn. Saved to `vial_rois.json`.

This is a one-time step — reuse the JSON for the same experimental setup.

In [5]:
ROI_JSON = os.path.join(OUTPUT_PATH, "vial_rois.json")
_vials = draw_and_save_vial_rois(video_path=str(PATH_TO_VID), roi_json_path=ROI_JSON)
save_run_params(OUTPUT_PATH, "roi", {k: list(v) for k, v in _vials.items()})

Saved ROIs to: ..\outputs\run_44\vial_rois.json


## 4 — RF-DETR + OC-SORT tracking

Runs the detector + tracker on every frame and writes a wide CSV.
This is the most time-consuming step. 

In [6]:
WIDE_CSV = os.path.join(OUTPUT_PATH, "tracks_wide_format.csv")
#added tracker bc tracker is now exported
df_wide, tracker = export_tracks_xy_tuple_csv_one_config(
    video_path=str(PATH_TO_VID),
    output_csv=WIDE_CSV,
    api_key=API_KEY,
    model_id=MODEL_ID,
    confidence=confidence,
    lost_track_buffer=lost_track_buffer,
    minimum_matching_threshold=min_matching_threshold,
    minimum_consecutive_frames=min_consecutive_frames,
    asso_func=asso_func,
    max_frames=None,
)

print(df_wide.shape)
save_run_params(OUTPUT_PATH, "tracker_output", {
    "wide_csv": WIDE_CSV, "frames": int(df_wide.shape[0]), "track_count": int(df_wide.shape[1] - 1),
})
df_wide.head()

Saved: ..\outputs\run_44\tracks_wide_format.csv  (frames=312, tracks=79)
(312, 80)


,frame,id1,id2,id3,id4,id5,id6,id7,id8,id9,...,id94,id95,id98,id101,id105,id107,id111,id112,id114,id116
0,0,"(454.89, 356.10)","(598.85, 328.52)","(422.60, 342.92)","(151.47, 394.52)","(285.33, 293.62)","(324.37, 395.17)","(421.88, 391.91)","(401.85, 352.12)","(136.99, 330.08)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,NaN,NaN,"(423.16, 340.32)","(151.68, 394.63)","(285.33, 293.69)","(324.39, 395.60)","(419.63, 394.23)","(401.55, 352.30)","(136.84, 330.14)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,NaN,NaN,"(422.28, 338.91)","(151.65, 394.32)","(285.07, 293.67)","(324.35, 395.59)","(419.09, 394.11)","(401.59, 352.09)","(136.86, 330.23)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,NaN,NaN,"(420.80, 335.83)","(152.42, 394.74)","(285.61, 293.25)","(325.15, 395.34)","(419.35, 394.10)","(401.63, 352.47)","(137.10, 330.43)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,NaN,NaN,"(419.62, 332.87)","(152.49, 394.79)","(287.08, 290.85)","(328.47, 393.75)","(419.41, 394.10)","(401.69, 352.35)","(137.01, 330.06)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Quick mid-pipeline check: are detections reaching the tracker?
# No compact IDs yet (stitching hasn't run), so no output report saved here.
run_diagnostics(
    tracker     = tracker,
    df_wide     = df_wide,
    df_stitched = None,
    n_expected  = 42,
    fps         = 30,
    config      = _cfg,
)

## 5 — Hungarian stitching

Links fragmented tracklets across gaps using motion-consistent assignment.
Output: long CSV with `orig_id` and `stitched_id` columns.

In [8]:
STITCHED_CSV = os.path.join(OUTPUT_PATH, "tracks_xy_stitched_long.csv")
LONG_CSV     = os.path.join(OUTPUT_PATH, "tracks_long_format.csv")

# Load vial ROIs
with open(ROI_JSON) as f:
    vial_rois = {k: tuple(v) for k, v in json.load(f).items()}

long_df   = wide_to_long(pd.read_csv(WIDE_CSV), out_csv=LONG_CSV)
tracklets = build_tracklets(long_df)

print(f"Built {len(tracklets)} tracklets from {long_df['orig_id'].nunique()} original IDs")

# Stitching — mode controlled by stitching_mode in config.yaml (default: per_vial)
stitched_df = stitch(
    long_df    = long_df,
    vial_rois  = vial_rois,
    tracklets  = tracklets,
    output_dir = OUTPUT_PATH,
)

stitched_df.to_csv(STITCHED_CSV, index=False)
print(f"\nSaved: {STITCHED_CSV}")
print(f"Stitched IDs: {stitched_df['stitched_id'].nunique()} (from {stitched_df['orig_id'].nunique()} original)")
save_run_params(OUTPUT_PATH, "stitching_output", {
    "stitched_csv": STITCHED_CSV,
    "stitched_ids": int(stitched_df["stitched_id"].nunique()),
    "original_ids": int(stitched_df["orig_id"].nunique()),
})

Built 79 tracklets from 79 original IDs
  vial1 round 1: 11 -> 6 IDs (cap 7)
  vial2 round 1: 12 -> 8 IDs (cap 7)
  vial2 round 2: 8 -> 7 IDs (cap 7)
  vial3 round 1: 13 -> 8 IDs (cap 7)
  vial3 round 2: 8 -> 7 IDs (cap 7)
  vial4 round 1: 12 -> 8 IDs (cap 7)
  vial4 round 2: 8 -> 7 IDs (cap 7)
  vial5 round 1: 13 -> 7 IDs (cap 7)
  vial6 round 1: 13 -> 8 IDs (cap 7)
  vial6 round 2: 8 -> 6 IDs (cap 7)

Saved: ..\outputs\run_44\tracks_xy_stitched_long.csv
Stitched IDs: 45 (from 79 original)


## 6 — Vial assignment + compact IDs

Assigns each point to a vial using the ROI JSON, then assigns compact sequential IDs
(left → right within each vial).

In [9]:
COMPACT_CSV = os.path.join(OUTPUT_PATH, "compact_tracks.csv")

df_compact = assign_vials_and_compact_ids(
    stitched_csv=STITCHED_CSV,
    roi_json=ROI_JSON,
    out_csv=COMPACT_CSV,
    fps=_s.get("fps", 60),
)

print(df_compact.shape)
save_run_params(OUTPUT_PATH, "compact", {"csv": COMPACT_CSV, "rows": int(df_compact.shape[0])})
df_compact.head()

(7632, 8)


,frame,orig_id,x,y,stitched_id,vial_id,compact_id,fps
0,0,id1,454.89,356.10,id1,vial4,26,30.0
1,0,id10,724.45,363.42,id10,vial6,42,30.0
2,1,id10,723.82,361.80,id10,vial6,42,30.0
3,2,id10,723.65,361.72,id10,vial6,42,30.0
4,3,id10,723.55,361.33,id10,vial6,42,30.0


In [ ]:
# Full diagnostics: all three stages with compact IDs after stitching.
# Saves metrics_report.md + two PNG plots to OUTPUT_PATH.
run_diagnostics(
    tracker     = tracker,
    df_wide     = df_wide,
    df_stitched = df_compact,
    df_compact  = df_compact,
    n_expected  = 42,
    fps         = 30,
    vial_rois   = vial_rois,
    config      = _cfg,
    output_dir  = OUTPUT_PATH,
)

## 7 — Overlay video

Renders each fly as a coloured dot on the original video.

In [ ]:
RAW_OVERLAY_MP4 = os.path.join(OUTPUT_PATH, f"{short_name}_overlay_raw_ocsort.mp4")
OVERLAY_MP4     = os.path.join(OUTPUT_PATH, f"{short_name}_overlay_vials_shaded.mp4")

render_raw_overlay_video(
    video_path=str(PATH_TO_VID),
    csv_path=LONG_CSV,
    out_mp4=RAW_OVERLAY_MP4,
)

render_vial_overlay_video(
    video_path=str(PATH_TO_VID),
    csv_path=COMPACT_CSV,
    out_mp4=OVERLAY_MP4,
)

save_run_params(OUTPUT_PATH, "outputs", {"raw_overlay": RAW_OVERLAY_MP4, "overlay": OVERLAY_MP4})
Video(RAW_OVERLAY_MP4, width=800)